# 가설 1 재실행 + 임계값 선택용 내부 validation (Colab)

**무엇을 하나**
1. 가설 1의 네 조건(KcELECTRA fine-tuning, seed 42·43·44 — 이전 가설 1 실행과 같은 seed)을 팀의 **기준 분할**(`results/split_assignment.csv`)에서 다시 학습해 test 행별 점수를 만든다.
2. 같은 fold의 train 안에서 내부 validation 점수를 만들어 판정 임계값을 고른다 (TF-IDF 기준선의 `OOF 조정`과 같은 절차, test 미사용).

**왜 다시 학습하나**: 가설 1의 첫 실행은 기준 분할과 다른 fold에서 학습되었다. scikit-learn 버전에 따라 `StratifiedGroupKFold(shuffle=True)`의 결과가 달라지기 때문이다. 내부 validation은 test 점수와 같은 fold 위에 있어야 하므로, 둘을 같은 분할에서 함께 만든다.

**분량**: seed 3개 기준 test 학습 60회(4조건 × 5 fold × 3 seed) + 내부 학습 240회. 이미 끝난 학습은 건너뛴다. 학습마다 걸린 시간이 출력된다. 앞선 가설 1 실행 기록(`cuda`)에서는 학습 1회가 25~47초였다.

**실행 전**: 메뉴 `런타임 → 런타임 유형 변경 → T4 GPU`. 위에서부터 차례로 실행한다. 세션이 끊기면 1~2번 셀을 다시 실행한 뒤 멈춘 셀부터 다시 실행하면 이어진다 (끝난 학습은 건너뜀).

## 1. 코드·데이터·모델 준비

In [ ]:
import os
if not os.path.exists("bda_teamproject"):
    !git clone -q -b refactor/unify-plm-eval https://github.com/contra333/bda_teamproject.git   # merge 뒤에는 -b main
%cd bda_teamproject
!pip install -q peft scikit-learn==1.9.1   # 분할이 scikit-learn 버전에 따라 달라지므로 requirements.txt 버전으로 맞춘다
!python -m src.download_data
from huggingface_hub import snapshot_download
snapshot_download("beomi/KcELECTRA-base-v2022", local_dir="model/KcELECTRA-base-v2022")
import sklearn, torch
print("scikit-learn", sklearn.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음 — 런타임 유형을 T4 GPU 로 바꾸세요")

## 2. 결과를 Google Drive에 바로 쌓기

결과 파일 세 개를 Drive에 두고 연결한다. 세션이 끊겨도 그때까지의 결과가 남고, 다시 연결하면 이어서 돈다. 저장소에 이미 있는 결과에서 이어 쓰므로, 끝난 학습은 건너뛴다. 처음부터 다시 돌리려면 아래 명령의 `--out`을 다른 이름으로 바꾼다.

In [ ]:
import os, shutil
from google.colab import drive
drive.mount("/content/drive")
SAVE = "/content/drive/MyDrive/kocoh_h1"
os.makedirs(SAVE, exist_ok=True)
for name in ["plm_h1.csv", "plm_h1_preds.csv", "plm_h1_inner_preds.csv"]:
    src, dst = f"results/{name}", f"{SAVE}/{name}"
    if not os.path.islink(src):
        if os.path.exists(src):            # 저장소에 있는 결과: Drive 에 없을 때만 옮겨 두고 거기서 이어 쓴다
            if not os.path.exists(dst):
                shutil.copy(src, dst)
            os.remove(src)
        os.symlink(dst, src)
    print(name, "— Drive 의 파일에 이어 씀" if os.path.exists(dst) else "— 새로 만듦")

## 3. 핵심 두 조건의 test 학습과 확인

실행을 시작하면 먼저 이 환경의 분할이 기준 분할 기록과 같은지 확인한다. `분할 확인: test fold 가 split_assignment.csv 기록과 같습니다`가 나와야 한다. `중단: ... 기준 분할 기록과 다릅니다`가 나오거나 1번 셀의 scikit-learn 설치가 실패하면, 우회하지 말고 거기서 멈춘 뒤 출력을 팀에 공유한다 (다른 분할로 학습하면 결과를 쓸 수 없다).

학습이 끝나면 같은 분할에서 돌린 10월 사전 실행(`results/plm_folds.csv`, Mac)과 fold별 macro-F1을 비교한다. 기기가 달라 완전히 같지는 않지만 fold별 값이 비슷해야 한다.

In [ ]:
!python src/plm.py --models KcELECTRA-base-v2022 --methods finetune --seeds 42 43 44 --out results/plm_h1.csv --resume --conditions Comment-only Context+Comment

import pandas as pd
new = pd.read_csv("results/plm_h1.csv").query("method == 'finetune' and seed == 42")
old = pd.read_csv("results/plm_folds.csv").query("model == 'KcELECTRA-base-v2022' and method == 'finetune'")
t = new.pivot(index="fold", columns="condition", values="macro_f1").join(
    old.pivot(index="fold", columns="condition", values="macro_f1"), lsuffix=" (이번)", rsuffix=" (10월)")
display(t.round(3)); print("평균:", t.mean().round(3).to_dict(), "| 기기:", new["device"].unique())

## 4. 핵심 두 조건의 내부 validation

In [ ]:
!python src/plm.py --models KcELECTRA-base-v2022 --methods finetune --seeds 42 43 44 --out results/plm_h1.csv --inner --conditions Comment-only Context+Comment

## 5. 나머지 두 조건 (글만, 따로 읽기)

In [ ]:
!python src/plm.py --models KcELECTRA-base-v2022 --methods finetune --seeds 42 43 44 --out results/plm_h1.csv --resume --conditions Comment-only Context+Comment Context-only Bi-encoder

In [ ]:
!python src/plm.py --models KcELECTRA-base-v2022 --methods finetune --seeds 42 43 44 --out results/plm_h1.csv --inner --conditions Context-only Bi-encoder

## 6. 평가와 결과 내려받기

기본(0.5) 행 옆에 `OOF 조정` 행이 나오고, 표 3-1이 임계값을 바꿨을 때의 변화를 보여 준다. 4번까지만 끝내고 실행해도 핵심 두 조건의 결과가 나온다.

In [ ]:
!python src/evaluate.py --preds results/plm_h1_preds.csv

In [ ]:
import glob, shutil
from google.colab import files
for name in ["plm_h1.csv", "plm_h1_preds.csv", "plm_h1_inner_preds.csv"]:
    if os.path.exists(f"results/{name}"):
        shutil.copy(f"results/{name}", f"/content/{name}")  # Drive 연결 파일의 사본
        files.download(f"/content/{name}")
for f in sorted(glob.glob("results/h1_*.csv")):
    files.download(f)

내려받은 `plm_h1*.csv` 세 파일을 저장소의 `results/`에 넣고 `python src/evaluate.py --preds results/plm_h1_preds.csv`를 실행하면 같은 표가 나온다. 사용한 기기는 `plm_h1.csv`의 `device` 열에 기록된다.